# Завдання
1. Провести парсинг самостійно обраного сайту. Вміст даних, що підлягають парсингу
– обрати самостійно(Autoria EV).
2. Результати парсингу зберегти у файлі. Тип файлу обрати самостійно.
3. Оцінити динаміку тренду реальних даних.
4. Здійснити визначення статистичних характеристик результатів парсингу.
5. Синтезувати та верифікувати модель даних, аналогічних за трендом і
статистичними характеристиками реальним даним, які є результатом парсингу.
6. Провести аналіз отриманих результатів.

# Imports

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.neighbors import KernelDensity
from PIL import Image
import json

# Site scrape

In [ ]:
# too much to scrape
# ev_df = None
# start_page = 1
# try:
#     ev_df = pd.read_excel('data/ev_cars.xlsx', index_col=0)
#     start_page = (ev_df.index[-1] // 100) + 1    # paginating by 100 in url
# except FileNotFoundError:
#     print('No ev_cars.xlsx, starting from 1 page')
#
# car_list = []
# for page in range(start_page, start_page + 10):
#     time.sleep(2)
#     try:
#         auto_ria_url = (
#             f'https://auto.ria.com/uk/search/?search_type=2&category=1&all[0].any[0]'
#             f'.fuel[0]=6&technical_condition[0]=1&technical_condition[1]=2&abroad=0&customs_cleared=1&order=7&page={page}&limit=100')
#         response = requests.get(auto_ria_url)
#         car_items = BeautifulSoup(response.content, 'html.parser')
#         soup = car_items.find_all('a', attrs={'class': 'link product-card horizontal'})
#         if not soup:
#             break
#         for i in soup:
#             try:
#                 car_title = i.find('div', attrs={'class': 'common-text size-16-20 titleS fw-bold mb-4'}).get_text()
#                 splitted_model = car_title.split()
#                 manufacturer, model, year = (splitted_model[0], ' '.join(splitted_model[1:-1]),
#                                              splitted_model[-1])
#                 car_details_link = i.attrs.get('href')
#                 car_details_response = requests.get(f'https://auto.ria.com{car_details_link}')
#                 car_detail_content = BeautifulSoup(car_details_response.content, 'html.parser')
#                 # date_added = i.find('span', attrs={'class': 'common-text footnote c-contrastSecondary'}).get_text()
#                 details_block = car_detail_content.find_all('div', attrs={'id': 'advertStatisticPublicationData'})
#                 date_added_text = (details_block[0].find('span', attrs={'class': 'common-text ws-pre-wrap body'})
#                                    .get_text())
#                 # split string with date range
#                 date_added = date_added_text.split()[-1]
#                 if date_added:
#                     price = (i.find('span', attrs={'class': 'common-text titleM c-green'}).get_text().replace('$',
#                                                                                                               '').replace(
#                         ' ', '')).strip()
#                     km_age = i.find('span', attrs={'class': 'common-text ellipsis-1 body'}).get_text().replace(' тис.'
#                                                                                                                ' км',
#                                                                                                                '000')
#                     car_list.append([manufacturer, model, year, price, km_age, date_added])
#             except Exception as e:
#                 continue
#         start_page += 1
#         print(start_page)
#     except Exception as e:
#         pass
#
# column_names = ['manufacturer', 'model', 'year', 'price', 'km_age', 'date_added']
# df = pd.DataFrame(car_list, columns=column_names)
# if ev_df is not None:
#     df = pd.concat([ev_df, df], ignore_index=True)
#
# # save to csv
# df.to_excel('data/ev_cars.xlsx')


# Read the file

In [ ]:
ev_df = pd.read_excel('data/ev_cars.xlsx')
# data look
print(ev_df.head(n=10))
print(ev_df.describe())
# get summary
print(ev_df.info())

## Load and basic cleaning

In [ ]:
"""Excel file: 'ev_cars.xlsx' with columns:
['manufacturer','model','year','price','date_added']
"""


def load_and_clean(path='data/ev_cars.xlsx', column_names=None):
    if column_names is None:
        column_names = ['manufacturer', 'model', 'year', 'price', 'km_age', 'date_added']
    df = pd.read_excel(path, names=None)
    # ensure required cols exist (try to coerce)
    df = df.rename(columns={c: c.strip() for c in df.columns})
    # keep only relevant columns if present
    for c in column_names:
        if c not in df.columns:
            raise ValueError(f"Missing column {c} in input file")
    df = df[column_names].copy()

    # types
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # price cleanup
    df['price'] = df['price'].astype(str).str.replace('[$, ]', '', regex=True)
    df['price'] = df['price'].str.replace('\xa0', '', regex=False)
    df['price'] = df['price'].str.replace(' ', '', regex=False)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df['km_age'] = pd.to_numeric(df['km_age'], errors='coerce')

    # drop rows with no date or no price
    df = df.dropna(subset=['date_added', 'price'])
    df = df.reset_index(drop=True)
    return df


# path to xlsx
path = 'data/ev_cars.xlsx'
ev_df = load_and_clean(path)
print("Loaded rows:", len(df))

## Trend dynamics

In [ ]:
def compute_trend(df, freq='ME', value_col='price', agg='median', plot=True):
    # aggregate by period (month)
    ts = getattr(df.set_index('date_added').resample(freq)[value_col], agg)()
    # linear trend fit (time -> value)
    ts_clean = ts.dropna()
    if len(ts_clean) < 3:
        raise ValueError("Not enough data points for trend estimation")
    X = np.arange(len(ts_clean)).reshape(-1, 1)
    y = ts_clean.values.reshape(-1, 1)
    lr = LinearRegression().fit(X, y)
    slope = float(lr.coef_[0])
    intercept = float(lr.intercept_)
    # seasonal decomposition (if monthly)
    try:
        decomp = seasonal_decompose(ts_clean, model='additive', period=12, extrapolate_trend='freq')
    except Exception:
        decomp = None
    if plot:
        plt.figure(figsize=(10, 4))
        plt.plot(ts.index, ts.values, marker='o', label=f'{agg} {value_col}')
        plt.plot(ts_clean.index, intercept + slope * np.arange(len(ts_clean)), '--',
                 label=f'Linear trend (slope={slope:.2f})')
        plt.title('Time series and linear trend')
        plt.legend()
        plt.tight_layout()
        plt.show()
    return {'series': ts, 'trend_slope_per_period': slope, 'trend_intercept': intercept, 'decomp': decomp}


# trend (median price per month)
trend_res = compute_trend(ev_df, freq='ME', value_col='price', agg='median', plot=True)

## Statistical characteristics

In [ ]:
def get_statistics(df, groupby_cols=None, numeric_cols=None):
    # overall describe
    if numeric_cols is None:
        numeric_cols = ['price', 'year']
    stats_overall = df[numeric_cols].describe().to_dict()
    # additional robust measures
    stats_overall['median'] = df[numeric_cols].median().to_dict()
    stats_overall['iqr'] = (df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25)).to_dict()
    # group-level
    group_stats = {}
    if groupby_cols:
        gp = df.groupby(groupby_cols)
        for name, g in gp:
            key = name if isinstance(name, str) else ','.join(map(str, name))
            group_stats[key] = g[numeric_cols].agg(['count', 'mean', 'median', 'std', 'min', 'max']).to_dict()
    return {'overall': stats_overall, 'by_group': group_stats}


# statistics
statistics = get_statistics(ev_df, groupby_cols=['manufacturer'], numeric_cols=['price', 'year'])
print("Overall stats snapshot:", json.dumps(statistics['overall'], default=str, indent=2))

## Synthesize synthetic dataset matching trend & stats

In [ ]:
def synthesize(df, n_samples=1000, freq='M'):
    # reproduce monthly counts trend and price distribution per-month
    # 1) compute counts per month
    counts = df.set_index('date_added').resample(freq)['price'].count().fillna(0)
    # fit KDE on log(price) to capture distribution shape
    prices = df['price'].dropna()
    logp = np.log(prices[prices > 0].values).reshape(-1, 1)
    kde = KernelDensity(kernel='gaussian', bandwidth=0.3).fit(logp)
    # sample months proportional to counts (preserving trend shape)
    # create a probability mass over observed months
    probs = counts / counts.sum() if counts.sum() > 0 else None
    months = counts.index
    if probs is None:
        # fallback: uniform over last 12 months
        months = pd.date_range(df['date_added'].min(), df['date_added'].max(), freq=freq)
        probs = np.ones(len(months)) / len(months)
    # sample months
    sampled_month_idx = np.random.choice(len(months), size=n_samples, p=probs)
    sampled_months = months[sampled_month_idx]
    # sample prices by sampling log-price from KDE and exponentiate
    sampled_log = kde.sample(n_samples=n_samples)
    sampled_price = np.exp(sampled_log).flatten()
    # assemble synthetic df
    synth = pd.DataFrame({
        'date_added': sampled_months,
        'price': sampled_price
    })
    # assign manufacturer/model/year by sampling conditional distributions from original
    # sample manufacturer by original frequency
    if 'manufacturer' in df.columns:
        manufacturers = df['manufacturer'].dropna().unique()
        m_probs = df['manufacturer'].value_counts(normalize=True).loc[manufacturers].values
        synth['manufacturer'] = np.random.choice(manufacturers, size=n_samples, p=m_probs)
    if 'year' in df.columns:
        years = df['year'].dropna().astype(int)
        year_probs = years.value_counts(normalize=True)
        year_choices = year_probs.index.astype(int).tolist()
        probs_years = year_probs.values
        synth['year'] = np.random.choice(year_choices, size=n_samples, p=probs_years)
    # basic rounding & types
    synth['price'] = synth['price'].round(2)
    synth = synth.reset_index(drop=True)
    return synth


# synthesize
synth = synthesize(ev_df, n_samples=2000, freq='ME')
synth.to_csv('data/synthetic_ev_cars.csv', index=False)
print("Synthetic saved to synthetic_ev_cars.csv")

## Validate synthetic vs real

In [ ]:
def validate_synthetic(real_df, synth_df):
    res = {}
    # compare moments for price
    for stat in ['mean', 'std', 'median', 'skew']:
        res[f'price_{stat}_real'] = getattr(real_df['price'].dropna(), stat)()
        res[f'price_{stat}_synth'] = getattr(synth_df['price'].dropna(), stat)()
    # KS test
    ks = stats.ks_2samp(real_df['price'].dropna(), synth_df['price'].dropna())
    res['ks_statistic'] = ks.statistic
    res['ks_pvalue'] = ks.pvalue
    # compare monthly counts correlation
    real_counts = real_df.set_index('date_added').resample('M')['price'].count().fillna(0)
    synth_counts = synth_df.set_index('date_added').resample('M')['price'].count().reindex(real_counts.index,
                                                                                           fill_value=0)
    res['monthly_counts_correlation'] = real_counts.corr(synth_counts)
    return res


# validate
val = validate_synthetic(ev_df, synth)
print("Validation metrics:", val)

## Math model for graphical transforms

In [ ]:
def affine_transform_matrix(scale=(1.0, 1.0), rotate_deg=0.0, translate=(0, 0), shear=(0.0, 0.0)):
    """
    Returns 3x3 affine matrix for 2D homogeneous coords:
    [ [s_x*cos - shear? etc], ... ]
    We'll build: Translate * Rotate * Shear * Scale
    """
    sx, sy = scale
    theta = np.deg2rad(rotate_deg)
    tx, ty = translate
    shx, shy = shear
    S = np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]])
    R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])
    H = np.array([[1, shx, 0], [shy, 1, 0], [0, 0, 1]])
    T = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])
    M = T @ R @ H @ S
    return M


def apply_affine_to_image(image_path, out_path, M, resample=Image.BICUBIC):
    img = Image.open(image_path)
    # PIL expects a 6-tuple (a,b,c,d,e,f) for forward affine mapping of output pixel (x,y) to input coords:
    # matrix = (a, b, c, d, e, f) corresponding to [[a,b,c],[d,e,f]]
    # We have a 3x3 M (homogeneous). Convert:
    a, b, c = M[0, 0], M[0, 1], M[0, 2]
    d, e, f = M[1, 0], M[1, 1], M[1, 2]
    transformed = img.transform(img.size, Image.AFFINE, data=(a, b, c, d, e, f), resample=resample)
    transformed.save(out_path)
    return out_path


# affine example for graphical object transform:
M = affine_transform_matrix(scale=(0.8, 0.8), rotate_deg=15, translate=(20, 10), shear=(0.1, 0.0))
# apply_affine_to_image('input.png','out.png',M)  # Uncomment if you have input.png
print("Affine matrix example:\n", M)